# Enhanced: LFM2 Fine-tuning with Periodic Evaluation and Weave Integration

> 📓 **For learning, not submission.** This walkthrough uses **Unsloth + bnb-4bit quantization** so you can iterate on a Colab T4 in ~15 min. The kit's **production submission path** is `./scripts/text/launch_hf_job.sh` → `scripts/text/train.py`, which uses **TRL + PEFT LoRA** (no Unsloth, no bnb) on a full-precision HF Jobs GPU. Different stacks, different checkpoint shapes — when you graduate, start fresh from the launcher; don't try to lift a notebook checkpoint into a launcher run.

This notebook enhances the original LFM2 fine-tuning notebook with the following features:

1. **Periodic Evaluation**: Evaluate model performance at each epoch or every N steps
2. **Weave Tracing**: Trace evaluation process with Weave for full observability
3. **W&B Models + Weave Integration**: Leverage the latest integration features
4. **Evaluation Visualization**: Progress tracking with radar charts and bar charts


## 1. Environment Setup and Installation


In [ ]:
import os

IS_COLAB_ENVIRONMENT = "COLAB_" in "".join(os.environ.keys())

In [ ]:
# Set directory if not in Colab
if not IS_COLAB_ENVIRONMENT:
    YOUR_INSTANCE_DIRECTORY = "~/liquid-ai-wandb-tokyo-hackathon"
    %cd {YOUR_INSTANCE_DIRECTORY}


In [ ]:
%%capture
import os

if IS_COLAB_ENVIRONMENT:
    # Load from Colab secrets
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_ENTITY"] = userdata.get('WANDB_ENTITY')
    os.environ["WANDB_PROJECT"] = userdata.get('WANDB_PROJECT')
else:
    # Load from local .env file
    %pip install python-dotenv ipywidgets
    from dotenv import load_dotenv

    load_dotenv()

WANDB_ENTITY = os.environ["WANDB_ENTITY"]
WANDB_PROJECT = os.environ["WANDB_PROJECT"]
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # log all model checkpoints during training



In [ ]:
%%capture
# Install required packages
%pip install trl==0.23.0 unsloth[colab-new]==2025.9.10 unsloth_zoo==2025.9.13
%pip install --upgrade weave wandb
%pip install nest-asyncio
%pip install structlog

## 2. Configuration and Imports


In [ ]:
# Project configuration
ORG_ENTITY = f"liquid-ai-org"
WANDB_REGISTRY_NAME = "liquid-ai-wandb-tokyo-hackathon"
WANDB_REGISTRY_COLLECTION_NAME = "LFM2-improved-ft"  # TODO(you): replace with your desired name
WANDB_MODEL_REGISTRY_BASENAME = f"wandb-registry-{WANDB_REGISTRY_NAME}/{WANDB_REGISTRY_COLLECTION_NAME}"

HF_PROFILE_NAME = "teozosa"  # TODO: Change to your HuggingFace profile name
HF_REPO_LFM2_FINETUNE = f"{HF_PROFILE_NAME}/{WANDB_REGISTRY_COLLECTION_NAME}"


In [ ]:
# Initialize our module-level logger
import structlog

LOGGER = structlog.get_logger()

In [ ]:
import wandb
import weave
# Must import unsloth before transformers, trl, torch, etc. to allow for proper patching of those libraries
import unsloth  # noqa: F401
import torch  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
from typing import Any, List, Dict, Optional  # noqa: F401
from collections import namedtuple
import asyncio
import json  # noqa: F401
from datetime import datetime
import nest_asyncio

# Enable asyncio in Jupyter environment
nest_asyncio.apply()

# Initialize W&B and Weave
wandb.login()
weave.init(f"{WANDB_ENTITY}/{WANDB_PROJECT}")


## 3. Custom Evaluation Callback Class


In [ ]:
from transformers import TrainerCallback
from transformers.trainer_utils import IntervalStrategy
import difflib


class WeaveEvaluationCallback(TrainerCallback):
    """
    Callback to run Weave Evaluation periodically and trace results
    """

    def __init__(self,
                 model_wrapper,
                 eval_dataset,
                 eval_interval: str = "epoch",  # "epoch" or "steps"
                 eval_steps: int = 100,
                 scorers: List = None,
                 max_eval_samples: int = 20):
        """
        Args:
            model_wrapper: Weave Model object
            eval_dataset: Dataset for evaluation
            eval_interval: Evaluation interval ("epoch" or "steps")
            eval_steps: Step interval for step-based evaluation
            scorers: List of scorer functions for evaluation
            max_eval_samples: Maximum number of samples to use for evaluation
        """
        self.model_wrapper = model_wrapper
        self.model_wrapper_base = copy.deepcopy(model_wrapper)
        self.eval_dataset = eval_dataset
        self.eval_interval = eval_interval
        self.eval_steps = eval_steps
        self.scorers = scorers or []
        self.max_eval_samples = max_eval_samples
        self.evaluation_results = []
        self.last_eval_step = 0
        self.total_callbacks_triggered = 0
        LOGGER.info(f"✅ WeaveEvaluationCallback initialized:", eval_interval=eval_interval,
                    eval_steps=f"{eval_steps if eval_interval == 'steps' else 'N/A (epoch-based)'}",
                    num_scorers=len(self.scorers), max_eval_samples=max_eval_samples,
                    eval_dataset_size=len(eval_dataset), model_wrapper_type=type(model_wrapper))

        # Get evaluation data subset (same size as Final Test Evaluation)
        eval_samples = min(self.max_eval_samples, len(self.eval_dataset))  # Match Final Test Evaluation sample size
        eval_subset = self.eval_dataset.select(range(eval_samples))

        # Prepare dataset in the same format as Final Test Evaluation
        self.eval_data = eval_subset.map(preprocess_dataset_for_weave_evaluation, batched=True,
                                         fn_kwargs={"base_model": self.model_wrapper_base}).to_list()

    @weave.op()
    async def run_evaluation(self, state):
        """Run evaluation and trace with Weave - same format as Final Test Evaluation"""

        LOGGER.debug(f"run_evaluation invoked", step=state.global_step, epoch=state.epoch)
        self.total_callbacks_triggered += 1

        # Use model/tokenizer from the Weave model wrapper
        model = self.model_wrapper.model
        # tokenizer = self.model_wrapper.tokenizer

        # Set model to evaluation mode
        was_training = model.training
        model.eval()

        # Create Weave Evaluation with same configuration
        evaluation = weave.Evaluation(
            dataset=self.eval_data,
            scorers=self.scorers,
            evaluation_name=f"Training Evaluation - Epoch {state.epoch:.1f} - Step {state.global_step}"
        )

        # Associate with current W&B run (same attributes as Final Test)
        with weave.attributes({
            'wandb-run-id': wandb.run.id if wandb.run else None,
            'evaluation_type': 'training_checkpoint',
            'global_step': state.global_step,
            'epoch': state.epoch,
            'training_loss': state.log_history[-1].get('loss', None) if state.log_history else None
        }):
            # Run evaluation
            try:
                summary, call = await evaluation.evaluate.call(evaluation, self.model_wrapper)
            except Exception as e:
                LOGGER.exception(f"Error in evaluation.evaluate.call", exc_info=e)
                raise

        # Save results
        result = {
            'step': state.global_step,
            'epoch': state.epoch,
            'timestamp': datetime.now().isoformat(),
            'summary': summary,
            'weave_call_id': call.id,
            'num_samples': len(self.eval_data)
        }
        self.evaluation_results.append(result)

        # Log to W&B with same format as Final Test
        if wandb.run:
            # Log individual metrics
            for metric, value in summary.items():
                if isinstance(value, dict) and 'mean' in value:
                    wandb.run.log({
                        f"training_eval/{metric}": value['mean']
                    }, step=state.global_step)

            # Also log the full normalized summary
            wandb.run.log({
                f"training_eval/{key}": value
                for key, value in pd.json_normalize(summary, sep='/').to_dict(orient="records")[0].items()
            }, step=state.global_step)

        # Print summary like Final Test Evaluation
        metric_value_dict = {f"{metric}": f"{value['mean']:.4f}" for metric, value in summary.items() if
                             isinstance(value, dict) and 'mean' in value}
        LOGGER.info(f"\n📊 Evaluation results", step={state.global_step}, **metric_value_dict)

        # Restore training mode if it was set
        if was_training:
            model.train()

        return result

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Called when trainer logs metrics - use this instead of on_step_end"""
        if logs and self.eval_interval == "steps":
            LOGGER.debug("on_log called", step=state.global_step, log_keys=(list(logs.keys()) if logs else "None"))

            # Check if we're at an evaluation step
            if (
                    state.global_step % self.eval_steps == 0 or
                    # if `logging_first_step=True` in the SFTTrainer, we also evaluate on the first training step
                    # to establish a baseline for comparison across evaluations at different time steps.
                    state.global_step == 1):
                if state.global_step != self.last_eval_step:  # Prevent duplicate evaluation
                    LOGGER.info(f"🔍 Running Weave evaluation...", step=state.global_step)
                    try:
                        loop = asyncio.get_event_loop()
                        result = loop.run_until_complete(
                            self.run_evaluation(state)
                        )
                        self.last_eval_step = state.global_step
                        LOGGER.info(f"✅ Evaluation completed", num_metrics_computed=len(result['summary']))
                    except Exception as e:
                        LOGGER.exception(f"❌ Error during evaluation", exc_info=e)
                        import traceback
                        traceback.print_exc()

    def on_epoch_end(self, args, state, control, **kwargs):
        """Process at epoch end"""
        if self.eval_interval == "epoch":
            LOGGER.info(f"🔍 Running Weave evaluation (epoch completed)", epoch=state.epoch)
            # Run async task synchronously
            try:
                loop = asyncio.get_event_loop()
                result = loop.run_until_complete(
                    self.run_evaluation(state)
                )
                LOGGER.info(f"✅ Evaluation completed", summary=result['summary'])
            except Exception as e:
                LOGGER.exception(f"❌ Error during evaluation", exc_info=e)
                import traceback
                traceback.print_exc()

    def on_evaluate(self, args, state, control, **kwargs):
        """Process during standard HF evaluation phase - this is called when eval_loss is computed"""
        LOGGER.info(f"🔍 Standard HF evaluation completed", step=state.global_step)

        # Run our custom Weave evaluation after HF evaluation
        if state.global_step != self.last_eval_step:  # Prevent duplicate if already run via on_log
            LOGGER.info(f"Running Weave evaluation...")
            try:
                loop = asyncio.get_event_loop()
                result = loop.run_until_complete(
                    self.run_evaluation(state)
                )
                self.last_eval_step = state.global_step
                LOGGER.info(f"✅ Weave evaluation completed", num_metrics_computed=len(result['summary']))
            except Exception as e:
                LOGGER.exception(f"❌ Error during Weave evaluation", exc_info=e)
                import traceback
                traceback.print_exc()

    def on_train_begin(self, args, state, control, **kwargs):
        self.on_evaluate(args, state, control, **kwargs)

    def on_train_end(self, args, state, control, **kwargs):
        """Display evaluation results summary at training end"""
        if self.evaluation_results:

            # Display summary metrics from each evaluation
            d = {
                f"{i + 1}": {"step": result['step'], "epoch": f"{result['epoch']:.2f})"}
                            | {metric: f"{value['mean']:.4f}" for metric, value in result['summary'].items() if
                               isinstance(value, dict) and 'mean' in value}
                for i, result in enumerate(self.evaluation_results)}
            LOGGER.info(f"📊 Evaluation Results Summary",
                        total_callbacks_triggered=self.total_callbacks_triggered,
                        final_last_eval_step=self.last_eval_step, num_evals_performed=len(self.evaluation_results),
                        **d)
            LOGGER.info("💡 Check Weave UI for automatic visualizations (radar charts, bar charts, traces)")
        else:
            LOGGER.warning("⚠️ No evaluation results collected during training")


## 4. Evaluation Scorer Functions


In [ ]:
# Define multiple evaluation metrics

@weave.op()
@weave.op()
async def conversation_scorer(output: str, reference_text: str) -> float:
    """Simple scoring function that checks the similarity between the assistant text in the dataset and the generated output from the model given the same conversation user message

    Args:
        output: model output for the given sample after fine-tuning
        reference_text: reference assistant response of the given sample

    Returns: Similarity ratio between the model output and assistant response in the text

    """

    return difflib.SequenceMatcher(None, output, reference_text).ratio()


async def conversation_improvement_scorer(output: str, base_model_output: str, reference_text: str) -> bool:
    """Simple scoring function that checks the difference between the assistant text in the dataset and the generated output from the model given the same conversation user message

    Args:
        output: model output for the given sample after fine-tuning
        base_model_output: model output for the given sample before fine-tuning
        reference_text: reference assistant response of the given sample

    Returns: If the similarity ratio between the model output and assistant response in the text is greater as compared to the ratio for the base model

    """
    similarity_ratio_output = await conversation_scorer(output, reference_text)
    similarity_ratio_baseline = await conversation_scorer(base_model_output, reference_text)
    return similarity_ratio_output > similarity_ratio_baseline


@weave.op()
async def response_length_scorer(output: str) -> float:
    """Evaluate response length appropriateness (0-1 score)"""
    ideal_length = 100  # Ideal response length
    actual_length = len(output.split())

    # Convert deviation from ideal length to score
    deviation = abs(actual_length - ideal_length) / ideal_length
    score = max(0, 1 - deviation)
    return score


@weave.op()
async def coherence_scorer(output: str) -> float:
    """Evaluate response coherence (simplified version)"""
    # Count sentences
    sentences = output.split('.')
    if len(sentences) < 2:
        return 0.5

    # Calculate word overlap between sentences (simple coherence metric)
    word_sets = [set(sent.lower().split()) for sent in sentences if sent.strip()]
    if len(word_sets) < 2:
        return 0.5

    total_overlap = 0
    comparisons = 0

    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            if word_sets[i] and word_sets[j]:
                overlap = len(word_sets[i] & word_sets[j]) / min(len(word_sets[i]), len(word_sets[j]))
                total_overlap += overlap
                comparisons += 1

    if comparisons > 0:
        return min(1.0, total_overlap / comparisons * 2)  # Scale adjustment
    return 0.5


@weave.op()
async def diversity_scorer(output: str) -> float:
    """Evaluate vocabulary diversity"""
    words = output.lower().split()
    if not words:
        return 0.0

    unique_words = set(words)
    diversity_ratio = len(unique_words) / len(words)
    return diversity_ratio


# Combine all scorers into a list
all_scorers = [
    conversation_scorer,
    conversation_improvement_scorer,
    response_length_scorer,
    coherence_scorer,
    diversity_scorer
]


## 5. Model Class and Training Configuration


In [ ]:
# Import code from the original notebook
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import FastModel
from peft import PeftModelForCausalLM
from huggingface_hub import HfApi
from pydantic import computed_field, Field
from transformers import Lfm2ForCausalLM, TextStreamer

# TrainConfig definition
TrainConfig = namedtuple("TrainConfig",
                         [
                             "per_device_train_batch_size",
                             "gradient_accumulation_steps",
                             "warmup_steps",
                             "num_train_epochs",
                             "max_steps",
                             "learning_rate",
                             "logging_steps",
                             "optim",
                             "weight_decay",
                             "lr_scheduler_type",
                             "seed",
                         ])


In [ ]:
import copy
import time
from typing import Any, List

import torch
import weave
from huggingface_hub import HfApi
from pydantic import computed_field, Field
from transformers import Lfm2ForCausalLM, TextStreamer

from unsloth import FastModel
from peft import PeftModelForCausalLM


class UnslothLoRALFM2(weave.Model):
    """
    We define an extra LFM2 class to store/version more parameters than just the model name.
    Especially relevant for fine-tuning (locally or aaS) because of specific parameters.
    """
    base_model: str
    revision: str | None

    # Initialization parameters
    is_training: bool
    r: int
    lora_alpha: int
    lora_dropout: float
    cm_temperature: float
    max_seq_length: int
    load_in_4bit: bool
    full_finetuning: bool
    inference_batch_size: int
    dtype: Any
    device: str

    # token = "hf_...", # use one if using gated models

    # Generation parameters
    # Recommended Liquid settings!
    max_new_tokens: int = Field(default=128,
                                description="[generation parameter] The maximum numbers of tokens to generate, ignoring the number of tokens in the prompt.")  # Increase for longer outputs!
    temperature: float = Field(default=0.3,
                               description="[generation parameter] The value used to modulate the next token probabilities.")
    min_p: float = Field(default=0.15,
                         description="[generation parameter] Minimum token probability, which will be scaled by the probability of the most likely token. It must be a value between 0 and 1. Typical values are in the 0.01-0.2 range, comparably selective as setting `top_p` in the 0.99-0.8 range (use the opposite of normal `top_p` values).")
    repetition_penalty: float = Field(default=1.05,
                                      description="[generation parameter]The parameter for repetition penalty. 1.0 means no penalty. See [this paper](https://huggingface.co/papers/1909.05858) for more details.")

    # Provenance
    model: Any = Field(default=None, exclude=True)  # Exclude from serialization as their reprs are noisy
    tokenizer: Any = Field(default=None, exclude=True)  # Exclude from serialization as their reprs are noisy

    def model_post_init(self, __context):

        # unsloth version (enable native 2x faster inference)
        if self.model is None or self.tokenizer is None:
            self.model, self.tokenizer = FastModel.from_pretrained(
                model_name=self.base_model,
                revision=self.revision,
                max_seq_length=self.max_seq_length,
                dtype=self.dtype,
                auto_model=Lfm2ForCausalLM,
                load_in_4bit=self.load_in_4bit,
                full_finetuning=self.full_finetuning,
            )

        # Training vs inference setup (may mutate tokenizer: pad token/chat template/etc.)
        if self.is_training:
            # If not full fine-tuning not already a LoRA model, add LoRA adapters
            if not self.full_finetuning and not isinstance(self.model, PeftModelForCausalLM):
                self.model = FastModel.get_peft_model(
                    self.model,
                    finetune_vision_layers=False,  # LFM for now is just text only
                    finetune_language_layers=True,  # Should leave on!
                    finetune_attention_modules=True,  # Attention good for GRPO
                    finetune_mlp_modules=True,  # Should leave on always!
                    r=self.r,
                    lora_alpha=self.lora_alpha,
                    lora_dropout=self.lora_dropout,
                    bias="none",
                    random_state=3407,
                )
            FastModel.for_training(self.model)
        else:
            FastModel.for_inference(self.model)

    @weave.op()
    async def predict(self, messages: List[str]) -> str:
        return self.predict_sync(messages=messages)

    @weave.op()
    def predict_sync(self, messages: list[str]) -> str:
        start_time = time.perf_counter()
        input_ids, output_ids, output = self._generate_response(messages)

        # Log usage information like OpenAI API spec
        prompt_tokens = input_ids.shape[1]
        completion_tokens = output_ids.shape[1] - input_ids.shape[1]
        end_time = time.perf_counter()
        latency_ms = (end_time - start_time) * 1000
        for k, v in {
            "usage.prompt_tokens": prompt_tokens,
            "usage.completion_tokens": completion_tokens,
            "usage.total_tokens": prompt_tokens + completion_tokens,
            "usage.latency_ms": latency_ms,
        }.items():
            weave.get_current_call().summary[k] = v

        return output

    def _generate_response(self, messages: list[str]) -> tuple[Any, Any, str]:
        # Include current hashes in Weave Trace
        input_ids = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,  # Must add for generation
            return_tensors="pt",
        ).to(self.device)

        output_ids = self.model.generate(
            input_ids=input_ids, max_new_tokens=self.max_new_tokens, use_cache=True, temperature=self.temperature,
            min_p=self.min_p
        )
        decoded_outputs = self.tokenizer.batch_decode(
            output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
        )
        output = "".join(decoded_outputs).strip()
        return input_ids, output_ids, output

    @weave.op()
    async def predict_stream(self, messages: List[str]) -> None:
        import time

        # Track timing
        start_time = time.perf_counter()

        inputs, output_ids = await self._generate_response_stream(messages)

        # Count prompt tokens
        prompt_tokens = inputs.shape[1] if hasattr(inputs, 'shape') else inputs['input_ids'].shape[1]
        # Count completion tokens (output excluding input)
        completion_tokens = output_ids.shape[1] - prompt_tokens
        # Track timing
        end_time = time.perf_counter()
        latency_ms = (end_time - start_time) * 1000

        # Log usage information like OpenAI API spec
        for k, v in {
            "usage.prompt_tokens": prompt_tokens,
            "usage.completion_tokens": completion_tokens,
            "usage.total_tokens": (prompt_tokens + completion_tokens),
            "usage.latency_ms": latency_ms,
        }.items():
            weave.get_current_call().summary[k] = v

    async def _generate_response_stream(self, messages: list[str]) -> tuple[Any, Any]:
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,  # Must add for generation
            return_tensors="pt",
            tokenize=True,
            return_dict=True,
        ).to(self.device)

        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            min_p=self.min_p,
            repetition_penalty=self.repetition_penalty,
            streamer=TextStreamer(self.tokenizer, skip_prompt=True),
        )
        return inputs, output_ids


## 6. Enhanced Training Function


In [ ]:
def get_trainer_with_evaluation_callback(
        model,
        tokenizer,
        config,
        train_dataset,
        eval_dataset,
        weave_model,
        eval_interval="epoch",
        eval_steps=100,
        max_eval_samples=20
) -> tuple:
    """
    Create trainer with evaluation callback
    """

    # Create base trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=config.per_device_train_batch_size,
            gradient_accumulation_steps=config.gradient_accumulation_steps,
            warmup_steps=config.warmup_steps,
            num_train_epochs=config.num_train_epochs,
            max_steps=config.max_steps,
            learning_rate=config.learning_rate,
            logging_steps=config.logging_steps,
            logging_first_step=True,
            optim=config.optim,
            weight_decay=config.weight_decay,
            lr_scheduler_type=config.lr_scheduler_type,
            seed=config.seed,
            report_to="wandb",
            output_dir="./results",  # Required parameter
            # Use eval_strategy instead of evaluation_strategy
            eval_strategy="steps" if eval_interval == "steps" else "epoch",
            eval_steps=eval_steps if eval_interval == "steps" else config.logging_steps,
            save_strategy="epoch",
            save_total_limit=3,
            load_best_model_at_end=False,  # Set to False to avoid issues with LoRA
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
    )

    # Apply train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    # Add Weave evaluation callback
    evaluation_callback = WeaveEvaluationCallback(
        model_wrapper=weave_model,
        eval_dataset=eval_dataset,
        eval_interval=eval_interval,
        eval_steps=eval_steps,
        scorers=all_scorers,
        max_eval_samples=max_eval_samples
    )

    trainer.add_callback(evaluation_callback)

    return trainer, evaluation_callback


## 7. Data Preparation Functions


In [ ]:
def get_messages_by_role(example, role: str) -> list[dict[str, str]]:
    return _get_messages_by_role(example["conversations"], role=role)


def _get_messages_by_role(conversations, role: str) -> list[dict[str, str]]:
    return [conversation for conversation in conversations if conversation["role"] == role]


def preprocess_datum(example, base_model: UnslothLoRALFM2):
    messages_user = get_messages_by_role(example, role="user")
    messages_assistant = get_messages_by_role(example, role="assistant")

    # To simplify evaluation, let's make the dataset single-turn chat.
    # Note that we assume the dataset conversations are properly formatted with user and assistant messages interleaved at each turn, with the user message preceding the assistant message.
    messages_user_first = messages_user[0]
    messages_assistant_first = messages_assistant[0]
    return {
        # Dataset must contain all inputs expected by the Model's `predict` method.
        "messages": [messages_user_first],

        # Fields needed for the `conversation_scorer` function.
        # Dataset rows must be a superset of fields expected by the Model's `predict` method.' and any downstream scoring functions used in the Weave Evaluation.

        "reference_text": messages_assistant_first["content"],
        # You can compose an eval dataset in multiple novel ways
        "base_model_output": base_model.predict_sync([messages_user_first])

    }


def preprocess_dataset_for_weave_evaluation_list(examples: list, base_model: UnslothLoRALFM2):
    """Preprocess dataset for Weave evaluation"""
    os.environ["WEAVE_PRINT_CALL_LINK"] = "false"
    preprocessed_data = [preprocess_datum(example, base_model) for example in examples]
    os.environ["WEAVE_PRINT_CALL_LINK"] = "true"
    return preprocessed_data


def preprocess_dataset_for_weave_evaluation(examples: dict, base_model: UnslothLoRALFM2):
    os.environ["WEAVE_PRINT_CALL_LINK"] = "false"

    """Preprocess dataset for Weave evaluation"""
    conversations = examples["conversations"]
    messages_user_firsts = [[_get_messages_by_role(conversation, role="user")[0]] for conversation in conversations]
    reference_texts = [_get_messages_by_role(conversation, role="assistant")[0]["content"] for conversation in
                       conversations]
    base_model_outputs = [base_model.predict_sync(messages) for messages in messages_user_firsts]

    os.environ["WEAVE_PRINT_CALL_LINK"] = "true"

    return {
        # Dataset must contain all inputs expected by the Model's `predict` method.
        "messages": messages_user_firsts,

        # Fields needed for the `conversation_scorer` function.
        # Dataset rows must be a superset of fields expected by the Model's `predict` method.' and any downstream scoring functions used in the Weave Evaluation.

        "reference_text": reference_texts,
        # You can compose an eval dataset in multiple novel ways
        "base_model_output": base_model_outputs

    }


def prepare_dataset(tokenizer, dataset_name="mlabonne/FineTome-100k-dedup", num_samples=3000):
    """Prepare and split dataset"""
    from datasets import load_dataset
    from unsloth.chat_templates import standardize_data_formats

    # Load dataset
    dataset = load_dataset(dataset_name, split=f"train[:{num_samples}]")

    # Standardize data format
    dataset = standardize_data_formats(dataset)

    # Apply chat template
    def formatting_prompts_func(examples):
        texts = tokenizer.apply_chat_template(
            examples["conversations"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

    dataset = dataset.map(formatting_prompts_func, batched=True)

    # Split into train/validation/test
    train_test = dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
    train_dataset = train_test["train"]
    test_dataset = train_test["test"]

    # Split validation set from training set
    train_val = train_dataset.train_test_split(test_size=0.1, shuffle=True, seed=42)
    train_dataset = train_val["train"]
    val_dataset = train_val["test"]

    return train_dataset, val_dataset, test_dataset


## 8. Main Execution Section


In [ ]:
# Model configuration

# TODO(you): specify which of the below models to use for finetuning
# More models at https://huggingface.co/unsloth
example_lfm2_models_base = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/LFM2-350M-unsloth-bnb-4bit",
    "unsloth/LFM2-700M-unsloth-bnb-4bit",
    "unsloth/LFM2-1.2B-unsloth-bnb-4bit",

    # Full 16bit unquantized models
    "unsloth/LFM2-350M",
    "unsloth/LFM2-700M",
    "unsloth/LFM2-1.2B",
]

# Initialize UnslothLoRALFM2 model

# "unsloth/LFM2-350M-unsloth-bnb-4bit"; If you already have a fine-tuned LFM2 in Huggingface, you can enter your repo here
base_model_repo = example_lfm2_models_base[0]
lfm2_ft = UnslothLoRALFM2(
    base_model=base_model_repo,
    revision=None,  # Set this to a specific Hugging Face commit, tag, etc. if you need a specific version
    is_training=True,
    r=16,  # Larger = higher accuracy, but might overfit
    lora_alpha=16,  # Recommended alpha == r at least
    lora_dropout=0,
    cm_temperature=1.0,
    max_seq_length=2048,  # Choose any for long context!
    load_in_4bit="4bit" in base_model_repo,  # Use 4bit quantization to reduce memory usage.
    full_finetuning=False,  # [NEW!] We have full finetuning now!
    inference_batch_size=2048,
    dtype=None,  # None for auto detection. (Float16 for Tesla T4, V100, Bfloat16 for Ampere+)
    device="cuda",
    # token = "hf_...", # use one if using gated models
)


In [ ]:
# Prepare datasets
dataset_name = "mlabonne/FineTome-100k-dedup"
train_dataset, val_dataset, test_dataset = prepare_dataset(
    tokenizer=lfm2_ft.tokenizer,
    dataset_name=dataset_name,
    num_samples=3000
)
LOGGER.info("Dataset stats", dataset_name=dataset_name, train_size=len(train_dataset), val_size=len(val_dataset),
            test_size=len(test_dataset))


In [ ]:
# Pre-training evaluation (baseline)
test_prompts = [
    "What is the capital of Japan?",
    "Write a poem about a sloth.",
    "Explain machine learning in simple terms."
]
for prompt in test_prompts:
    response = await lfm2_ft.predict([{"role": "user", "content": prompt}])
    LOGGER.info(f"🔍 Pre-training baseline evaluation...", input=prompt,
                output_truncated=response[:200])  # Display first 200 characters

In [ ]:
# Training configuration
default_configs_dict = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 5,
    "num_train_epochs": 1,  # Set to 1 epoch for testing
    "max_steps": 100,  # Limit to 100 steps for testing (remove or set to -1 for full training)
    "learning_rate": 2e-4,
    "logging_steps": 10,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 3407,
}

train_configs = TrainConfig(**default_configs_dict)
LOGGER.info(f"📊 Training configuration", **train_configs._asdict())


In [ ]:
# Initialize W&B run
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config=default_configs_dict,
    name="LFM2-fine-tune", # Comment-out to auto-generate unique run names
    tags=["periodic-evaluation", "weave-integration"]
)


In [ ]:
# Create trainer with evaluation callback
# Estimate total training steps (respecting max_steps if set)
if train_configs.max_steps != -1:
    total_steps = train_configs.max_steps
else:
    total_steps = len(train_dataset) // (
            train_configs.per_device_train_batch_size * train_configs.gradient_accumulation_steps) * max(1,
                                                                                                         train_configs.num_train_epochs)
total_steps = max(1, total_steps)
eval_steps_interval = max(10, total_steps // 5)  # Aim for ~5 evaluations during training

trainer, evaluation_callback = get_trainer_with_evaluation_callback(
    model=lfm2_ft.model,
    tokenizer=lfm2_ft.tokenizer,
    config=train_configs,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    weave_model=lfm2_ft,
    eval_interval="steps",  # Changed to steps for more control
    eval_steps=eval_steps_interval,  # Dynamic based on dataset size
    max_eval_samples=10  # Match Final Test Evaluation sample size
)

LOGGER.info("✅ Trainer ready",
            total_steps=total_steps,
            eval_steps_interval=eval_steps_interval,
            expected_num_evals=total_steps // eval_steps_interval,
            max_eval_samples=evaluation_callback.max_eval_samples,
            num_registered_callbacks=len(trainer.callback_handler.callbacks))

In [ ]:
# Display memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
LOGGER.info("GPU info", gpu=gpu_stats.name, max_memory_gb=max_memory, reserved_memory_gb=start_gpu_memory)


In [ ]:
# Run training
LOGGER.info("🚀 Starting training...", weave_eval_config={"eval_steps_interval": evaluation_callback.eval_steps,
                                                         "num_eval_metrics": len(evaluation_callback.scorers),
                                                         "registered_callbacks": [type(cb).__name__ for cb in
                                                                                  trainer.callback_handler.callbacks]})

lfm2_base = copy.deepcopy(lfm2_ft)  # Keep a copy of the baseline model for later reference
trainer_stats = trainer.train()

LOGGER.info("✅ Training complete!", num_evals=len(evaluation_callback.evaluation_results))
if len(evaluation_callback.evaluation_results) == 0:
    LOGGER.warning("⚠️ No Weave evaluations were performed. Checking callback registration...",
                   is_callback_trainer=(evaluation_callback in trainer.callback_handler.callbacks),
                   num_steps_trained=(trainer_stats.global_step if hasattr(trainer_stats, 'global_step') else 'N/A'),
                   eval_steps_expected=evaluation_callback.eval_steps
                   )


In [ ]:
# Display training statistics
def show_final_memory_and_time_stats(trainer_stats):
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3)
    lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

    LOGGER.info("Training stats.",
                train_runtime_seconds=trainer_stats.metrics['train_runtime'],
                train_runtime_minutes=round(trainer_stats.metrics['train_runtime'] / 60, 2),
                used_memory_gb=used_memory,
                used_memory_for_lora_gb=used_memory_for_lora,
                used_percentage_max_memory=used_percentage,
                used_percentage_max_memory_for_lora=lora_percentage
                )


show_final_memory_and_time_stats(trainer_stats)


In [ ]:
# Run final evaluation
LOGGER.info("\n🔍 Running final test set evaluation...")
# Final evaluation on test set
test_subset = test_dataset.select(range(min(10, len(test_dataset))))
final_evaluation = weave.Evaluation(
    dataset=test_subset.map(preprocess_dataset_for_weave_evaluation, batched=True,
                            fn_kwargs={"base_model": lfm2_base}).to_list(),
    scorers=all_scorers,
    evaluation_name="Final Test Evaluation"
)

with weave.attributes({'wandb-run-id': run.id, 'evaluation_type': 'final_test'}):
    final_summary, final_call = await final_evaluation.evaluate.call(final_evaluation, lfm2_ft)

metric_value_dict = {f"{metric}": f"{value['mean']:.4f}" for metric, value in final_summary.items() if
                     isinstance(value, dict) and 'mean' in value}
LOGGER.info(f"\n📊 Final　Evaluation results", step="final", **metric_value_dict)

# Log to W&B
wandb.log(
    {f"final_test/{k}": v for k, v in pd.json_normalize(final_summary, sep='/').to_dict(orient="records")[0].items()})


In [ ]:
# Test inference with trained model
test_prompts = [
    "What is the capital of Japan?",
    "Write a poem about a sloth.",
    "Explain machine learning in simple terms."
]

for prompt in test_prompts:
    response = await lfm2_ft.predict([{"role": "user", "content": prompt}])
    LOGGER.info(f"🤖 Testing trained model responses...", input=prompt,
                output_truncated=response[:200])  # Display first 200 characters


In [ ]:
# Detailed analysis of evaluation results
if evaluation_callback.evaluation_results:

    # Convert to DataFrame
    df_eval = pd.DataFrame(evaluation_callback.evaluation_results)

    # Calculate improvement rate for each metric
    if len(df_eval) > 1:
        first_eval = df_eval.iloc[0]['summary']
        last_eval = df_eval.iloc[-1]['summary']

        print("\nImprovement rates:")
        improvement_rates = {}
        for metric in first_eval.keys():
            if metric in last_eval:
                if isinstance(first_eval[metric], dict) and 'mean' in first_eval[metric]:
                    first_val = first_eval[metric]['mean']
                    last_val = last_eval[metric]['mean']
                    improvement = ((last_val - first_val) / abs(first_val)) * 100 if first_val != 0 else 0
                    improvement_rates[metric] = f"{improvement:.2f}%"
                    print(f"  {metric}: {improvement:+.2f}%")

        LOGGER.info("\n📈 Detailed analysis of evaluation results:", improvement_rates=improvement_rates)


In [ ]:
# Finish W&B run
wandb.finish()